In [1]:
import polars as pl
import numpy as np
from pathlib import Path
from project_paths import paths
from imd_features.config import FeatureSetConfig, GroupConfig, ReductionMethod
from imd_features.process import create_feature_set
from imd_features.evaluate import evaluate
from imd_features.diagnostic import group_summary

In [2]:
input_df = pl.read_parquet(paths.input_file)
input_df.shape

(268, 439)

In [3]:
# basically a better thought through version of the mixed example 
# connectivity factor analysis with 8 factors
# OSM PCA with 8 components
# all uc features are selected and in crime the ones that have really low occurance across lsoas are excluded
calibrated_reduction_v2 = FeatureSetConfig(
    name="calibrated_reduction_v2",
    description="v2: expanded land registry FA(5), nearest-POI, updated ratios",
    groups={
        "crime": GroupConfig(
            columns=[
                "violent-crime",
                "burglary",
                "anti-social-behaviour",
                "shoplifting",
                "criminal-damage-arson",
                "drugs",
                "robbery",
                "vehicle-crime",
                "total_crimes",
                "resolution_rate",
            ],
            scale=True,
        ),
        "uc": GroupConfig(
            columns=[
                "total_claims",
                "mean_monthly_claims",
                "%_claims_nwr",
                "%_claims_planfw",
                "%_claims_prepfw",
                "%_claims_sfw",
            ],
            scale=True,
        ),
        "connectivity": GroupConfig(
            columns=[
                "Employment (walking)",
                "Education (walking)",
                "Healthcare (walking)",
                "Leisure and Community (walking)",
                "Shopping (walking)",
                "Residential (walking)",
                "Overall (walking)",
                "Employment (cycling)",
                "Education (cycling)",
                "Healthcare (cycling)",
                "Leisure and Community (cycling)",
                "Shopping (cycling)",
                "Residential (cycling)",
                "Overall (cycling)",
                "Business (public transport)",
                "Education (public transport)",
                "Healthcare (public transport)",
                "Leisure and Community (public transport)",
                "Shopping (public transport)",
                "Residential (public transport)",
                "Overall (public transport)",
                "Employment (driving)",
                "Education (driving)",
                "Healthcare (driving)",
                "Leisure and Community (driving)",
                "Shopping (driving)",
                "Residential (driving)",
                "Overall (driving)",
                "Employment (overall)",
                "Education (overall)",
                "Healthcare (overall)",
                "Leisure and Community (overall)",
                "Shopping (overall)",
                "Residential (overall)",
                "Overall",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=8,
        ),
        "land_registry": GroupConfig(
            columns=[
                "lsoa_mean_price",
                "lsoa_median_price",
                "lsoa_stdev_price",
                "lsoa_max_price",
                "lsoa_min_price",
                "lsoa_range_price",
                "lsoa_price_inequality",
                "T_mean_price",
                "F_mean_price",
                "S_mean_price",
                "D_mean_price",
                "O_mean_price",
                "total_transactions",
                "T_count_transactions",
                "F_count_transactions",
                "S_count_transactions",
                "D_count_transactions",
                "O_count_transactions",
                "new_build_proportion",
                "freehold_proportion",
                "terraced_proportion",
                "flats_proportion",
                "semi_detached_proportion",
                "detached_proportion",
                "other_proportion",
                "new_build_price_premium",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=5,
        ),
        "osm_amenities": GroupConfig(
            columns=[
                f"count_{group}_{dist}"
                for group in [
                    "healthcare_access",
                    "education_skills",
                    "financial_services",
                    "food_dining",
                    "fast_food_takeaway",
                    "alcohol_gambling",
                    "transport_public",
                    "essential_services",
                    "childcare_early_years",
                    "retail_commerce",
                    "community_social",
                    "social_support",
                ]
                for dist in [500, 1000]
            ],
            scale=True,
            reduction_method=ReductionMethod.PCA,
            n_components=8,
        ),
        "nearest_poi": GroupConfig(
            columns=[
                "nearest_shop",
                "nearest_hospital",
                "nearest_pharmacy",
                "nearest_school",
                "nearest_kindergarten",
                "nearest_college",
                "nearest_university",
                "nearest_bank",
                "nearest_atm",
                "nearest_fast_food",
                "nearest_pub",
                "nearest_bar",
                "nearest_gambling",
                "nearest_cinema",
                "nearest_theatre",
                "nearest_social_facility",
                "nearest_bicycle_parking",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4,
        ),
        "osm_ratios": GroupConfig(
            columns=[
                "ratio_fast_food_takeaway_to_food_dining_1000",
                "ratio_alcohol_gambling_to_cultural_entertainment_1000",
                "ratio_alcohol_gambling_to_financial_services_1000",
                "ratio_fast_food_takeaway_to_healthcare_access_1000",
                "ratio_sustainable_transport_to_transport_car_1000",
                "ratio_education_skills_to_alcohol_gambling_1000",
                "ratio_healthcare_access_to_alcohol_gambling_1000",
                "ratio_essential_services_to_alcohol_gambling_1000",
                "ratio_childcare_early_years_to_alcohol_gambling_1000",
            ],
            scale=True,
        ),
        "landuse_environment": GroupConfig(
            columns=[
                "landuse_residential_0",
                "landuse_commercial_0",
                "landuse_industrial_0",
                "landuse_grass_0",
                "landuse_recreation_ground_0",
                "landuse_education_0",
                "streetlit_percentage",
            ],
            scale=True,
        ),
        "population": GroupConfig(
            columns=[
                "lsoa_population",
                "aged_under_15",
                "working_age_population",
                "pension_age_population",
            ],
            scale=True,
        ),
    },
)

In [4]:
# OSM amenities at 3 meaningful scales (walking distance, short drive, trip)
DEPRIVATION_AMENITY_GROUPS = [
    "healthcare_access",
    "education_skills",
    "food_dining",
    "fast_food_takeaway",
    "essential_services",
    "transport_public",
    "financial_services",
    "childcare_early_years",
    "retail_commerce",
    "social_support",
    "community_social",
]

multiscale_access_v2 = FeatureSetConfig(
    name="multiscale_access_v2",
    description="v2: adds nearest-POI, ratios, and expanded land registry",
    groups={
        "walkable_500": GroupConfig(
            columns=[f"count_{g}_500" for g in DEPRIVATION_AMENITY_GROUPS],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=3,
        ),
        "neighbourhood_1000": GroupConfig(
            columns=[f"count_{g}_1000" for g in DEPRIVATION_AMENITY_GROUPS],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=3,
        ),
        "district_2500": GroupConfig(
            columns=[f"count_{g}_2500" for g in DEPRIVATION_AMENITY_GROUPS],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=3,
        ),
        "nearest_poi": GroupConfig(
            columns=[
                "nearest_shop",
                "nearest_hospital",
                "nearest_pharmacy",
                "nearest_school",
                "nearest_kindergarten",
                "nearest_college",
                "nearest_university",
                "nearest_bank",
                "nearest_atm",
                "nearest_fast_food",
                "nearest_pub",
                "nearest_bar",
                "nearest_gambling",
                "nearest_cinema",
                "nearest_theatre",
                "nearest_social_facility",
                "nearest_bicycle_parking",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4,
        ),
        "osm_ratios": GroupConfig(
            columns=[
                "ratio_fast_food_takeaway_to_food_dining_1000",
                "ratio_alcohol_gambling_to_cultural_entertainment_1000",
                "ratio_alcohol_gambling_to_financial_services_1000",
                "ratio_fast_food_takeaway_to_healthcare_access_1000",
                "ratio_sustainable_transport_to_transport_car_1000",
                "ratio_education_skills_to_alcohol_gambling_1000",
                "ratio_healthcare_access_to_alcohol_gambling_1000",
                "ratio_essential_services_to_alcohol_gambling_1000",
                "ratio_childcare_early_years_to_alcohol_gambling_1000",
            ],
            scale=True,
        ),
        "economic": GroupConfig(
            columns=[
                "total_claims",
                "mean_monthly_claims",
                "%_claims_nwr",
                "%_claims_planfw",
                "%_claims_prepfw",
                "%_claims_sfw",
            ],
            scale=True,
        ),
        "land_registry": GroupConfig(
            columns=[
                "lsoa_mean_price",
                "lsoa_median_price",
                "lsoa_stdev_price",
                "lsoa_max_price",
                "lsoa_min_price",
                "lsoa_range_price",
                "lsoa_price_inequality",
                "T_mean_price",
                "F_mean_price",
                "S_mean_price",
                "D_mean_price",
                "O_mean_price",
                "total_transactions",
                "T_count_transactions",
                "F_count_transactions",
                "S_count_transactions",
                "D_count_transactions",
                "O_count_transactions",
                "new_build_proportion",
                "freehold_proportion",
                "terraced_proportion",
                "flats_proportion",
                "semi_detached_proportion",
                "detached_proportion",
                "other_proportion",
                "new_build_price_premium",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=5,
        ),
        "crime": GroupConfig(
            columns=[
                "violent-crime",
                "burglary",
                "anti-social-behaviour",
                "shoplifting",
                "criminal-damage-arson",
                "drugs",
                "total_crimes",
                "resolution_rate",
            ],
            scale=True,
        ),
        "connectivity_summary": GroupConfig(
            columns=[
                "Overall (walking)",
                "Overall (cycling)",
                "Overall (public transport)",
                "Overall (driving)",
                "Overall",
            ],
            scale=True,
        ),
        "landuse": GroupConfig(
            columns=[
                "landuse_residential_0",
                "landuse_commercial_0",
                "landuse_industrial_0",
                "landuse_grass_0",
                "landuse_recreation_ground_0",
                "landuse_education_0",
            ],
            scale=True,
        ),
        "population": GroupConfig(
            columns=[
                "lsoa_population",
                "aged_under_15",
                "working_age_population",
                "pension_age_population",
            ],
            scale=True,
        ),
    },
)

In [5]:
# Create population rate features
raw = pl.read_parquet(paths.input_file)
engineered = raw.with_columns(
    (pl.col("total_crimes") / pl.col("lsoa_population") * 1000).alias("crime_rate_per_1000"),
    (pl.col("violent-crime") / pl.col("lsoa_population") * 1000).alias("violent_crime_rate"),
    (pl.col("anti-social-behaviour") / pl.col("lsoa_population") * 1000).alias("asb_rate"),
    (pl.col("burglary") / pl.col("lsoa_population") * 1000).alias("burglary_rate"),
    (pl.col("drugs") / pl.col("lsoa_population") * 1000).alias("drugs_rate"),
    (pl.col("total_claims") / pl.col("working_age_population")).alias("uc_claim_rate"),
    (pl.col("total_nwr_claims") / pl.col("working_age_population")).alias("uc_nwr_rate"),
    (pl.col("aged_under_15") / pl.col("lsoa_population")).alias("youth_share"),
    (pl.col("pension_age_population") / pl.col("lsoa_population")).alias("elderly_share"),
    (pl.col("total_transactions") / pl.col("lsoa_population") * 1000).alias("transactions_per_capita"),
)

engineered_path = paths.input_file.parent / "combined_engineered.parquet"
engineered.write_parquet(engineered_path)
del raw, engineered

In [6]:
# population rates instead of raw counts
engineered_rates_v2 = FeatureSetConfig(
    name="engineered_rates_v2",
    description="v2: population-normalised rates with expanded land registry and nearest-POI",
    groups={
        "crime_rates": GroupConfig(
            columns=[
                "crime_rate_per_1000",
                "violent_crime_rate",
                "asb_rate",
                "burglary_rate",
                "drugs_rate",
                "resolution_rate",
            ],
            scale=True,
        ),
        "benefit_rates": GroupConfig(
            columns=[
                "uc_claim_rate",
                "uc_nwr_rate",
                "%_claims_planfw",
                "%_claims_sfw",
            ],
            scale=True,
        ),
        "access": GroupConfig(
            columns=[
                "Overall",
                "Overall (walking)",
                "Overall (public transport)",
                "Healthcare (walking)",
            ],
            scale=True,
        ),
        "nearest_poi": GroupConfig(
            columns=[
                "nearest_shop",
                "nearest_hospital",
                "nearest_pharmacy",
                "nearest_school",
                "nearest_bank",
                "nearest_fast_food",
                "nearest_pub",
                "nearest_gambling",
            ],
            scale=True,
        ),
        "housing": GroupConfig(
            columns=[
                "lsoa_mean_price",
                "lsoa_median_price",
                "lsoa_price_inequality",
                "lsoa_max_price",
                "transactions_per_capita",
                "flats_proportion",
                "terraced_proportion",
                "detached_proportion",
                "new_build_proportion",
                "freehold_proportion",
            ],
            scale=True,
        ),
        "services_1000": GroupConfig(
            columns=[
                "count_healthcare_access_1000",
                "count_education_skills_1000",
                "count_essential_services_1000",
                "count_transport_public_1000",
                "count_financial_services_1000",
                "count_retail_commerce_1000",
                "count_fast_food_takeaway_1000",
                "count_food_dining_1000",
            ],
            scale=True,
        ),
        "osm_ratios": GroupConfig(
            columns=[
                "ratio_fast_food_takeaway_to_food_dining_1000",
                "ratio_alcohol_gambling_to_financial_services_1000",
                "ratio_fast_food_takeaway_to_healthcare_access_1000",
                "ratio_sustainable_transport_to_transport_car_1000",
                "ratio_education_skills_to_alcohol_gambling_1000",
            ],
            scale=True,
        ),
        "environment": GroupConfig(
            columns=[
                "landuse_residential_0",
                "landuse_industrial_0",
                "streetlit_percentage",
                "landuse_grass_0",
                "landuse_commercial_0",
            ],
            scale=True,
        ),
        "demographics": GroupConfig(
            columns=[
                "lsoa_population",
                "youth_share",
                "elderly_share",
                "working_age_population",
            ],
            scale=True,
        ),
    },
)

In [7]:
configs = {
    "calibrated_reduction": (calibrated_reduction_v2, paths.input_file),
    "multiscale_access": (multiscale_access_v2, paths.input_file),
    "engineered_rates": (engineered_rates_v2, engineered_path),
}

all_results = {}
all_metadata = {}

for name, (config, input_path) in configs.items():
    df, metadata = create_feature_set(input_path, config)
    results = evaluate(df, config)
    n_features = df.shape[1] - 1
    all_results[name] = {"n_features": n_features, **results}
    all_metadata[name] = (config, metadata)

In [8]:
configs_created = {
    "calibrated_reduction": calibrated_reduction_v2,
    "multiscale_access": multiscale_access_v2,
    "engineered_rates": engineered_rates_v2,
}

results = {}
for name, config in configs_created.items():
    df = pl.read_parquet(paths.output / f"{config.output_name}.parquet")
    results[name] = evaluate(df, config)

In [ ]:
rows = []
for name, result in results.items():
    for model_name, metrics in result.items():
        rows.append({
            "config": name,
            "model": model_name,
            "r2_mean": metrics["r2_mean"],
            "rmse_mean": metrics["rmse_mean"],
            "spearman_mean": metrics["spearman_mean"],
        })

pl.DataFrame(rows).sort("r2_mean", descending=True)

config,model,r2_mean,rmse_mean,spearman_mean
str,str,f64,f64,f64
"""engineered_rates""","""random_forest""",0.872597,5.574342,0.936976
"""calibrated_reduction""","""random_forest""",0.820038,6.690891,0.91166
"""multiscale_access""","""random_forest""",0.809125,6.896092,0.913759
"""engineered_rates""","""ridge""",0.769828,7.45147,0.887774
"""calibrated_reduction""","""ridge""",0.701739,8.496371,0.855907
"""multiscale_access""","""ridge""",0.636079,9.286118,0.82935


In [10]:
for name, result in results.items():
    gi = pl.DataFrame(result["random_forest"]["group_importance"])
    print(f"\n{name} (RF r2={result['random_forest']['r2_mean']:.4f})")
    print(gi.sort("total_importance", descending=True))


calibrated_reduction (RF r2=0.8200)
shape: (9, 4)
┌─────────────────────┬──────────────────┬─────────────────┬────────────┐
│ group               ┆ total_importance ┆ mean_importance ┆ n_features │
│ ---                 ┆ ---              ┆ ---             ┆ ---        │
│ str                 ┆ f64              ┆ f64             ┆ i64        │
╞═════════════════════╪══════════════════╪═════════════════╪════════════╡
│ uc                  ┆ 0.607091         ┆ 0.101182        ┆ 6          │
│ landuse_environment ┆ 0.103904         ┆ 0.014843        ┆ 7          │
│ connectivity        ┆ 0.085855         ┆ 0.010732        ┆ 8          │
│ population          ┆ 0.04811          ┆ 0.012028        ┆ 4          │
│ crime               ┆ 0.04341          ┆ 0.004341        ┆ 10         │
│ land_registry       ┆ 0.035206         ┆ 0.007041        ┆ 5          │
│ osm_amenities       ┆ 0.027912         ┆ 0.003489        ┆ 8          │
│ nearest_poi         ┆ 0.026631         ┆ 0.006658        ┆ 

In [12]:
reduced_configs = {
    "calibrated_reduction": calibrated_reduction_v2,
    "multiscale_access": multiscale_access_v2,
    "engineered_rates": engineered_rates_v2,
}

for name, config in reduced_configs.items():
    df = pl.read_parquet(paths.output / f"{config.output_name}.parquet")
    _, metadata = create_feature_set(
        paths.input_file if name != "engineered_rates" else engineered_path,
        config,
    )
    print(f"\n{name}")
    print(group_summary(config, metadata))


calibrated_reduction
shape: (9, 7)
┌──────────────┬──────────────┬──────────────┬────────┬──────────────┬──────────────┬──────────────┐
│ group        ┆ input_featur ┆ output_featu ┆ scaled ┆ reduction    ┆ mean_noise_v ┆ explained_va │
│ ---          ┆ es           ┆ res          ┆ ---    ┆ ---          ┆ ar           ┆ r            │
│ str          ┆ ---          ┆ ---          ┆ bool   ┆ str          ┆ ---          ┆ ---          │
│              ┆ i64          ┆ i64          ┆        ┆              ┆ f64          ┆ f64          │
╞══════════════╪══════════════╪══════════════╪════════╪══════════════╪══════════════╪══════════════╡
│ crime        ┆ 10           ┆ 10           ┆ true   ┆ none         ┆ null         ┆ null         │
│ uc           ┆ 6            ┆ 6            ┆ true   ┆ none         ┆ null         ┆ null         │
│ connectivity ┆ 35           ┆ 8            ┆ true   ┆ factor_analy ┆ 0.0349       ┆ null         │
│              ┆              ┆              ┆        ┆

In [13]:
best_name = pl.DataFrame(rows).filter(
    pl.col("model") == "random_forest"
).sort("r2_mean", descending=True)["config"][0]

fi = pl.DataFrame(results[best_name]["random_forest"]["feature_importance"])
fi.sort("importance_mean", descending=True).head(20)

feature,group,importance_mean,importance_std
str,str,f64,f64
"""uc_claim_rate""","""benefit_rates""",0.373929,0.015648
"""uc_nwr_rate""","""benefit_rates""",0.370839,0.013481
"""youth_share""","""demographics""",0.043252,0.013434
"""transactions_per_capita""","""housing""",0.031272,0.024854
"""lsoa_max_price""","""housing""",0.026593,0.013654
…,…,…,…
"""lsoa_price_inequality""","""housing""",0.003579,0.001354
"""landuse_residential_0""","""environment""",0.003282,0.001237
"""nearest_pharmacy""","""nearest_poi""",0.003209,0.001511


In [14]:
er_fi = pl.DataFrame(results["engineered_rates"]["random_forest"]["feature_importance"])
er_fi.filter(
    pl.col("group").is_in(["crime_rates", "benefit_rates", "demographics"])
).sort("importance_mean", descending=True)

feature,group,importance_mean,importance_std
str,str,f64,f64
"""uc_claim_rate""","""benefit_rates""",0.373929,0.015648
"""uc_nwr_rate""","""benefit_rates""",0.370839,0.013481
"""youth_share""","""demographics""",0.043252,0.013434
"""violent_crime_rate""","""crime_rates""",0.015529,0.003992
"""working_age_population""","""demographics""",0.007024,0.002055
…,…,…,…
"""resolution_rate""","""crime_rates""",0.002442,0.000433
"""elderly_share""","""demographics""",0.002229,0.000763
"""burglary_rate""","""crime_rates""",0.002032,0.000418
